# 🧠 SYNAPSE-R1 V2 — ALINHAMENTO COGNITIVO MASSIVO (22.000+ AMOSTRAS)
### Arquitetura: 1.21B Parâmetros | Sparse MoE (12x Experts Top-2)
### Desenvolvido do Zero por **Nandinho**

Este notebook executa o treinamento de alinhamento com **raciocínio profundo estilo DeepSeek-R1** em grande escala (22.000 amostras completas em português), acelerado com **AMP BFloat16 na GPU NVIDIA A100-SXM4-80GB**.

In [ ]:
# 1. Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive conectado com sucesso!')

In [ ]:
# 2. Clonar ou Atualizar o Repositório Oficial
import os
if not os.path.exists('/content/cerebro-ia-102m'):
    !git clone https://github.com/nandinhotv/cerebro-ia-102m.git /content/cerebro-ia-102m
%cd /content/cerebro-ia-102m
!git pull origin main
print('✅ Repositório atualizado na versão mais recente!')

In [ ]:
# 3. Verificar GPU e Instalar Dependências
!nvidia-smi
!pip install -q tokenizers torch datasets
print('✅ Ambiente configurado!')

In [ ]:
# 4. Disparar o Alinhamento Cognitivo Massivo R1 V2 (22.000 Amostras | 3 Épocas)
!python alinhar_synapse_1b_r1_massivo.py

In [ ]:
# 5. Chat Interativo Livre no Colab com a Synapse R1 V2
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from modelo_synapse_1b import BrainMoE1B, Config1B

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok = Tokenizer.from_file('tokenizer.json')
cfg = Config1B(vocab_size=tok.get_vocab_size())
model = BrainMoE1B(cfg).to(device)

final_ckpt = '/content/drive/MyDrive/Synapse-1B/checkpoints/synapse-1b-r1-v2-final.pt'
raw = torch.load(final_ckpt, map_location='cpu', weights_only=False)
st = raw.get('model_state_dict', raw.get('model_state', raw))
model.load_state_dict({k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k: v for k, v in st.items()})
model.eval()
print('🎉 Synapse R1 V2 carregada e pronta para conversar!')

def perguntar(texto):
    prompt = f'### Usuário:\n{texto}\n\n### Assistente:\n'
    ids = tok.encode(prompt).ids
    gen = list(ids)
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available(), dtype=torch.bfloat16):
            for _ in range(400):
                curr = torch.tensor([gen[-1024:]], device=device)
                logits, _ = model(curr)
                nxt_logits = logits[0, -1, :].clone()
                nxt_logits[0] = -float('inf')
                v, _ = torch.topk(nxt_logits, 40)
                nxt_logits[nxt_logits < v[-1]] = -float('inf')
                probs = F.softmax(nxt_logits / 0.6, dim=-1)
                nxt = torch.multinomial(probs, 1).item()
                if nxt == 2: break
                gen.append(nxt)
    print(tok.decode(gen[len(ids):]).strip())

# Exemplo de teste rápido
perguntar('Quem é você e qual a sua arquitetura?')